# V-JEPA 2 UCF101 Kaggle Training
Make sure to:
1. Select Accelerator: **GPU T4 x2**
2. Add the UCF101 dataset to your Kaggle notebook (search for `ucf101` in Kaggle datasets)
3. Upload this entire `vjepa2` folder to Kaggle, or git clone it inside the Kaggle notebook.

In [ ]:
!pip install uv
!uv pip install --system -r requirements.txt
!uv pip install --system decord pandas scikit-learn

In [ ]:
import os
!mkdir -p /kaggle/working/checkpoints
if not os.path.exists('/kaggle/working/checkpoints/vitl.pt'):
    !wget -O /kaggle/working/checkpoints/vitl.pt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt

In [ ]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

# UPDATE THIS PATH if your kaggle dataset is named differently
dataset_dir = "/kaggle/input/ucf101/UCF101/UCF-101"

if not os.path.exists(dataset_dir):
    print(f"Warning: {dataset_dir} not found. Please mount the UCF101 dataset.")
else:
    classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
    class_to_idx = {cls: i for i, cls in enumerate(classes)}
    
    data = []
    for cls in classes:
        cls_dir = os.path.join(dataset_dir, cls)
        for video_file in glob.glob(os.path.join(cls_dir, "*.avi")):
            data.append((video_file, class_to_idx[cls]))
            
    df = pd.DataFrame(data, columns=["path", "label"])
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    
    train_df.to_csv("/kaggle/working/ucf101_train_paths.csv", index=False, header=False, sep=" ")
    val_df.to_csv("/kaggle/working/ucf101_val_paths.csv", index=False, header=False, sep=" ")
    print(f"Created CSVs with {len(train_df)} training and {len(val_df)} validation videos.")

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

# Chạy file eval/main.py, nó sẽ tự động nhận diện multiprocessing với các GPUs được chỉ định
!python evals/main.py --fname configs/eval/vitl/ucf101.yaml --devices cuda:0 cuda:1